In [8]:
from __future__ import annotations
import importlib.metadata as metadata
import json
import os
import random
import re
import sys
from pathlib import Path
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    import faiss
except ImportError as e:
    raise ImportError(
        "Не найден пакет faiss"
    ) from e

try:
    import torch
except ImportError:
    torch = None

SEED = 42
TOP_K_CHUNKS = 5
TOP_K_SOURCES = 3

HW14_DIR = Path.cwd()
DATA_DIR = HW14_DIR / "data"
ARTIFACTS_DIR = HW14_DIR / "artifacts"

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

def get_version(package_name: str) -> str:
    try:
        return metadata.version(package_name)
    except Exception:
        return "not installed"

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

set_seed(SEED)

if torch is not None and torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# Вывод информации специально для копирования в report.md
print("Среда и воспроизводимость\n")
print(f"Python: {sys.version.split()[0]}")
print(f"faiss / faiss-cpu: {get_version('faiss-cpu')}")
print(f"sentence-transformers / transformers / sklearn: {get_version('sentence-transformers')} / {get_version('transformers')} / {get_version('scikit-learn')}")
print(f"torch (если использовался): {get_version('torch')}")
print(f"Устройство (CPU/GPU): {DEVICE.upper()}")
print(f"Seed: {SEED}")

Среда и воспроизводимость

Python: 3.9.13
faiss / faiss-cpu: 1.13.0
sentence-transformers / transformers / sklearn: 5.1.2 / 4.57.6 / 1.6.1
torch (если использовался): 2.6.0+cu124
Устройство (CPU/GPU): CUDA
Seed: 42


In [16]:
# --- База знаний и первичный анализ (Пункт 2.3.2) ---



# Формируем корпус текстов
primary_articles = [
    {"id": "DOC-01", "topic": "architecture", "name": "Трансформеры", "body": "Архитектура нейронных сетей, представленная в 2017 году компанией Google. Трансформеры полностью отказались от рекуррентных слоев в пользу механизма внутреннего внимания (self-attention), что позволило эффективно распараллеливать вычисления и обрабатывать длинные тексты."},
    {"id": "DOC-02", "topic": "embeddings", "name": "Эмбеддинги", "body": "Плотные векторные представления данных в многомерном пространстве. Близкие по смыслу тексты имеют похожие векторы (косинусное расстояние между ними стремится к единице). Эмбеддинги позволяют алгоритмам машинного обучения 'понимать' семантику."},
    {"id": "DOC-03", "topic": "search", "name": "FAISS", "body": "Библиотека от Facebook AI Similarity Search для быстрого поиска сходства и кластеризации плотных векторов. Она содержит алгоритмы, которые ищут в наборах векторов любого размера. Часто используется в RAG-системах для быстрого извлечения контекста."},
    {"id": "DOC-04", "topic": "rag", "name": "RAG", "body": "Retrieval-Augmented Generation — подход, объединяющий информационный поиск и генерацию текста. Вместо того чтобы полагаться только на знания в весах LLM, система RAG сначала ищет релевантные факты во внешней базе, а затем передает их языковой модели."},
    {"id": "DOC-05", "topic": "errors", "name": "Галлюцинации LLM", "body": "Явление, при котором языковая модель генерирует правдоподобный, но фактически неверный текст. Использование RAG-архитектуры является одним из лучших способов снижения количества галлюцинаций, так как модель опирается на поданный ей контекст."},
    {"id": "DOC-06", "topic": "nlp", "name": "Токенизация", "body": "Процесс разбиения сырого текста на токены: слова, части слов или символы. Современные LLM часто используют алгоритмы токенизации подслов, такие как BPE (Byte Pair Encoding), чтобы эффективно справляться с редкими словами и опечатками."},
    {"id": "DOC-07", "topic": "training", "name": "Fine-tuning", "body": "Процесс дообучения предварительно обученной нейронной сети под конкретную задачу. Популярны методы PEFT (например, LoRA), которые обучают лишь малую часть весов, экономя видеопамять и время вычислений."},
    {"id": "DOC-08", "topic": "nlp", "name": "Prompt Engineering", "body": "Процесс составления и оптимизации запросов (промптов) для получения нужного результата от LLM. Включает в себя подходы Zero-shot, Few-shot и Chain-of-Thought (просьба к модели рассуждать пошагово)."},
    {"id": "DOC-09", "topic": "search", "name": "Векторные БД", "body": "Специализированные базы данных для хранения и поиска многомерных векторов. В отличие от SQL-баз, которые ищут точные текстовые совпадения, векторные БД выполняют семантический поиск, находя ближайших соседей по смыслу."},
    {"id": "DOC-10", "topic": "preprocessing", "name": "Чанкинг", "body": "Разделение длинных документов на более короткие фрагменты (чанки) перед их векторизацией. Это нужно из-за ограничений на длину контекста у моделей эмбеддингов. Часто чанки делают с перекрытием (overlap)."}
]

# Дополнительные тексты для шага с переиндексацией
reserve_articles = [
    {"id": "DOC-11", "topic": "search", "name": "Гибридный поиск", "body": "Метод, объединяющий векторный семантический поиск (например, через FAISS) и классический лексический поиск (например, BM25 по ключевым словам). Это позволяет находить документы, которые совпадают и по смыслу, и по точным терминам."},
    {"id": "DOC-12", "topic": "search", "name": "Reranking", "body": "Процесс дополнительной сортировки результатов, полученных от векторного поиска. Обычно выполняется с помощью кросс-энкодеров (cross-encoders), которые медленнее, но гораздо точнее оценивают релевантность между запросом и каждым найденным чанком."},
    {"id": "DOC-13", "topic": "rag", "name": "Self-RAG", "body": "Продвинутая техника, при которой языковая модель сама решает, нужно ли ей обращаться к внешней базе знаний для ответа на вопрос, и сама оценивает качество найденного контекста (релевантен он или нет)."}
]

path_main_kb = DATA_DIR / "kb_initial.json"
path_extra_kb = DATA_DIR / "kb_update.json"

with open(path_main_kb, "w", encoding="utf-8") as f:
    json.dump(primary_articles, f, ensure_ascii=False, indent=4)

with open(path_extra_kb, "w", encoding="utf-8") as f:
    json.dump(reserve_articles, f, ensure_ascii=False, indent=4)

# 3. Загружаем и проверяем
with open(path_main_kb, "r", encoding="utf-8") as f:
    loaded_main = json.load(f)

with open(path_extra_kb, "r", encoding="utf-8") as f:
    loaded_extra = json.load(f)

print("Статистика базы знаний:")
print(f" -> Базовый набор загружен: {len(loaded_main)} док.")
print(f" -> Набор для обновления подготовлен: {len(loaded_extra)} док.")
print(f" -> Общий объем составит: {len(loaded_main) + len(loaded_extra)} док.\n")

df_knowledge = pd.DataFrame(loaded_main)

display(Markdown("### Структура документов (Превью метаданных)"))
display(df_knowledge.drop(columns=['body']).head())

display(Markdown("### Примеры содержания текстов (3 документа)"))
for index, row in df_knowledge.head(3).iterrows():
    print(f"[{row['id']}] {row['name']} (Тематика: {row['topic']})")
    print(f"Текст: {row['body']}\n")

display(Markdown("""
> **Обоснование выбора базы знаний:**
> Данный корпус представляет собой выжимку технических терминов из области ИИ. Использование пайплайна Retrieval / mini-RAG на таких данных крайне эффективно, так как позволяет предоставлять языковой модели строгие определения фактов (например, точное описание FAISS или механизм Attention). Это сводит риск возможных «галлюцинаций» LLM к минимуму.
"""))

Статистика базы знаний:
 -> Базовый набор загружен: 10 док.
 -> Набор для обновления подготовлен: 3 док.
 -> Общий объем составит: 13 док.



### Структура документов (Превью метаданных)

,id,topic,name
0,DOC-01,architecture,Трансформеры
1,DOC-02,embeddings,Эмбеддинги
2,DOC-03,search,FAISS
3,DOC-04,rag,RAG
4,DOC-05,errors,Галлюцинации LLM


### Примеры содержания текстов (3 документа)

[DOC-01] Трансформеры (Тематика: architecture)
Текст: Архитектура нейронных сетей, представленная в 2017 году компанией Google. Трансформеры полностью отказались от рекуррентных слоев в пользу механизма внутреннего внимания (self-attention), что позволило эффективно распараллеливать вычисления и обрабатывать длинные тексты.

[DOC-02] Эмбеддинги (Тематика: embeddings)
Текст: Плотные векторные представления данных в многомерном пространстве. Близкие по смыслу тексты имеют похожие векторы (косинусное расстояние между ними стремится к единице). Эмбеддинги позволяют алгоритмам машинного обучения 'понимать' семантику.

[DOC-03] FAISS (Тематика: search)
Текст: Библиотека от Facebook AI Similarity Search для быстрого поиска сходства и кластеризации плотных векторов. Она содержит алгоритмы, которые ищут в наборах векторов любого размера. Часто используется в RAG-системах для быстрого извлечения контекста.




> **Обоснование выбора базы знаний:**
> Данный корпус представляет собой выжимку технических терминов из области ИИ. Использование пайплайна Retrieval / mini-RAG на таких данных крайне эффективно, так как позволяет предоставлять языковой модели строгие определения фактов (например, точное описание FAISS или механизм Attention). Это сводит риск возможных «галлюцинаций» LLM к минимуму.


In [17]:
# --- Чанкинг документов (Пункт 2.3.3) ---


def create_overlapping_chunks(text: str, window_size: int = 90, overlap: int = 20) -> List[str]:
    """
    Разбивает текст на фрагменты (чанки) методом скользящего окна.
    window_size: максимальная длина чанка в символах.
    overlap: количество символов, которые повторяются между соседними чанками.
    """
    if len(text) <= window_size:
        return [text]
        
    chunks = []
    start_idx = 0
    stride = window_size - overlap 
    
    while start_idx < len(text):
        end_idx = start_idx + window_size
        chunk = text[start_idx:end_idx]
        chunks.append(chunk)
        
        if end_idx >= len(text):
            break
            
        start_idx += stride
        
    return chunks

CHUNK_SIZE = 90
OVERLAP = 20

processed_chunks = []

for article in loaded_main:
    text_fragments = create_overlapping_chunks(article["body"], window_size=CHUNK_SIZE, overlap=OVERLAP)
    
    for i, fragment in enumerate(text_fragments):
        processed_chunks.append({
            "chunk_id": f"{article['id']}_ch{i+1}",
            "parent_id": article['id'],
            "topic": article['topic'],
            "text": fragment
        })

print("Чанкинг завершен.")
print(f"Из {len(loaded_main)} документов получено {len(processed_chunks)} текстовых фрагментов.\n")


demo_parent_id = "DOC-04" 
demo_doc = next(item for item in loaded_main if item["id"] == demo_parent_id)

display(Markdown(f"### Демонстрация чанкинга для документа `{demo_parent_id}` ({demo_doc['name']})"))
print(f"Исходный текст ({len(demo_doc['body'])} символов):\n{demo_doc['body']}\n")

print("Полученные фрагменты:")
demo_chunks = [c for c in processed_chunks if c["parent_id"] == demo_parent_id]
for c in demo_chunks:
    print(f"[{c['chunk_id']}] (Длина: {len(c['text'])}): {c['text']}")

display(Markdown(f"""
**Обоснование выбранных параметров (`chunk_size`={CHUNK_SIZE}, `overlap`={OVERLAP}):**
* **Размер чанка ({CHUNK_SIZE} символов)** позволяет разбить словарные статьи на атомарные смысловые части (по 1-2 предложения). Это увеличивает точность векторного поиска, так как в вектор не смешиваются разные концепции из одного текста. Также это позволяет выполнить требование к объему базы (получить более 30 чанков из 10 документов).
* **Перекрытие ({OVERLAP} символов)** сохраняет связность на границах разреза. Если термин или важная фраза разрывается, перекрытие гарантирует, что хотя бы в одном из соседних чанков эта фраза останется целой для корректного построения эмбеддинга.
"""))

Чанкинг завершен.
Из 10 документов получено 36 текстовых фрагментов.



### Демонстрация чанкинга для документа `DOC-04` (RAG)

Исходный текст (250 символов):
Retrieval-Augmented Generation — подход, объединяющий информационный поиск и генерацию текста. Вместо того чтобы полагаться только на знания в весах LLM, система RAG сначала ищет релевантные факты во внешней базе, а затем передает их языковой модели.

Полученные фрагменты:
[DOC-04_ch1] (Длина: 90): Retrieval-Augmented Generation — подход, объединяющий информационный поиск и генерацию тек
[DOC-04_ch2] (Длина: 90): оиск и генерацию текста. Вместо того чтобы полагаться только на знания в весах LLM, систем
[DOC-04_ch3] (Длина: 90):  в весах LLM, система RAG сначала ищет релевантные факты во внешней базе, а затем передает
[DOC-04_ch4] (Длина: 40): зе, а затем передает их языковой модели.



**Обоснование выбранных параметров (`chunk_size`=90, `overlap`=20):**
* **Размер чанка (90 символов)** позволяет разбить словарные статьи на атомарные смысловые части (по 1-2 предложения). Это увеличивает точность векторного поиска, так как в вектор не смешиваются разные концепции из одного текста. Также это позволяет выполнить требование к объему базы (получить более 30 чанков из 10 документов).
* **Перекрытие (20 символов)** сохраняет связность на границах разреза. Если термин или важная фраза разрывается, перекрытие гарантирует, что хотя бы в одном из соседних чанков эта фраза останется целой для корректного построения эмбеддинга.


In [18]:
# --- Эмбеддинги и индекс FAISS (Пункт 2.3.4) ---


#  Загрузка embedding-модели
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
print(f"Загрузка модели {MODEL_NAME}...")

embedder = SentenceTransformer(MODEL_NAME, device=DEVICE)

texts_to_embed = [chunk["text"] for chunk in processed_chunks]
print(f"Вычисление эмбеддингов для {len(texts_to_embed)} фрагментов...")

vector_embeddings = embedder.encode(texts_to_embed, convert_to_numpy=True)

faiss.normalize_L2(vector_embeddings)

#  Построение индекса FAISS
embedding_dim = vector_embeddings.shape[1]
vector_index = faiss.IndexFlatIP(embedding_dim)
vector_index.add(vector_embeddings)

print("Индекс FAISS успешно построен.")
print(f"Размерность векторов: {embedding_dim}")
print(f"Всего векторов в индексе: {vector_index.ntotal}\n")


sample_queries = [
    "В чем суть механизма внутреннего внимания?",
    "Как можно бороться с выдумками языковых моделей?",
    "Для чего нужна библиотека от Facebook?",
    "Что такое BPE и зачем оно нужно?"
]

SEARCH_TOP_K = 3

display(Markdown(f"### Примеры поиска (Top-{SEARCH_TOP_K})"))

for query in sample_queries:
    print(f"Запрос: «{query}»")
    
    # Векторизуем и нормализуем запрос точно так же, как базу
    q_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    
    distances, indices = vector_index.search(q_vec, SEARCH_TOP_K)
    
    for rank in range(SEARCH_TOP_K):
        chunk_idx = indices[0][rank]
        score = distances[0][rank]
        retrieved_chunk = processed_chunks[chunk_idx]
        
        print(f"  [{rank+1}] Сходство: {score:.3f} | Документ: {retrieved_chunk['topic']} (ID: {retrieved_chunk['chunk_id']})")
        print(f"      Текст: {retrieved_chunk['text']}")
    print("-" * 60)

Загрузка модели paraphrase-multilingual-MiniLM-L12-v2...
Вычисление эмбеддингов для 36 фрагментов...
Индекс FAISS успешно построен.
Размерность векторов: 384
Всего векторов в индексе: 36



### Примеры поиска (Top-3)

Запрос: «В чем суть механизма внутреннего внимания?»
  [1] Сходство: 0.808 | Документ: architecture (ID: DOC-01_ch3)
      Текст: еханизма внутреннего внимания (self-attention), что позволило эффективно распараллеливать 
  [2] Сходство: 0.495 | Документ: errors (ID: DOC-05_ch4)
      Текст: рается на поданный ей контекст.
  [3] Сходство: 0.437 | Документ: training (ID: DOC-07_ch1)
      Текст: Процесс дообучения предварительно обученной нейронной сети под конкретную задачу. Популярн
------------------------------------------------------------
Запрос: «Как можно бороться с выдумками языковых моделей?»
  [1] Сходство: 0.823 | Документ: errors (ID: DOC-05_ch1)
      Текст: Явление, при котором языковая модель генерирует правдоподобный, но фактически неверный тек
  [2] Сходство: 0.729 | Документ: rag (ID: DOC-04_ch4)
      Текст: зе, а затем передает их языковой модели.
  [3] Сходство: 0.617 | Документ: embeddings (ID: DOC-02_ch4)
      Текст: о обучения 'понимать' семантику.
-------------

In [19]:
# --- Контрольные запросы и оценка retrieval (Пункт 2.3.5) ---


# 1. Подготовка контрольных запросов (10 штук)
# Формат: (Текст запроса, Ожидаемый ID документа)
benchmark_queries = [
    ("В каком году придумали Трансформеры и кто их автор?", "DOC-01"),
    ("Как алгоритмы понимают смысл слов в многомерном пространстве?", "DOC-02"),
    ("Какая библиотека лучше всего подходит для кластеризации векторов?", "DOC-03"),
    ("Как расшифровывается аббревиатура RAG?", "DOC-04"),
    ("Что означает термин галлюцинация в контексте LLM?", "DOC-05"),
    ("Приведи пример популярного алгоритма токенизации подслов.", "DOC-06"),
    ("В чем заключается метод LoRA?", "DOC-07"),
    ("Что такое Chain-of-Thought и для чего он нужен?", "DOC-08"),
    ("Чем векторные базы данных отличаются от обычных SQL?", "DOC-09"),
    ("Зачем делать нахлест (overlap) при нарезке документов?", "DOC-10")
]

EVAL_TOP_K = 3
evaluation_results = []
hits_count = 0

print(f"Запуск оценки retrieval (Top-{EVAL_TOP_K})...")

# 2. Выполнение оценки
for query_text, expected_doc_id in benchmark_queries:
    q_vec = embedder.encode([query_text], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    
    dists, idxs = vector_index.search(q_vec, EVAL_TOP_K)
    
    retrieved_parent_ids = [processed_chunks[i]['parent_id'] for i in idxs[0]]
    
    is_hit = 1 if expected_doc_id in retrieved_parent_ids else 0
    hits_count += is_hit
    
    recall_at_k = 1.0 if is_hit == 1 else 0.0
    
    rank = -1
    for r, pid in enumerate(retrieved_parent_ids):
        if pid == expected_doc_id:
            rank = r + 1
            break
            
    evaluation_results.append({
        "query": query_text,
        "expected_source": expected_doc_id,
        "retrieved_sources": ", ".join(retrieved_parent_ids),
        "hit_at_k": is_hit,
        "recall_at_k": recall_at_k,
        "rank_of_first_relevant": rank
    })

# 3. Анализ метрик
df_eval = pd.DataFrame(evaluation_results)
overall_hit_rate = hits_count / len(benchmark_queries)

display(Markdown("### Результаты оценки Retrieval"))
print(f"Общая метрика Hit@{EVAL_TOP_K}: {overall_hit_rate:.2f} ({(overall_hit_rate*100):.0f}% успешных извлечений)")
print(f"Общая метрика Recall@{EVAL_TOP_K}: {overall_hit_rate:.2f}\n")

display(df_eval)

# 4. Сохранение артефакта (Пункт 4.1)
eval_artifact_path = ARTIFACTS_DIR / "retrieval_eval.csv"
df_eval.to_csv(eval_artifact_path, index=False, encoding="utf-8")

print(f"\nАртефакт сохранен: {eval_artifact_path}")

Запуск оценки retrieval (Top-3)...


### Результаты оценки Retrieval

Общая метрика Hit@3: 1.00 (100% успешных извлечений)
Общая метрика Recall@3: 1.00



,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant
0,В каком году придумали Трансформеры и кто их а...,DOC-01,"DOC-01, DOC-02, DOC-01",1,1.0,1
1,Как алгоритмы понимают смысл слов в многомерно...,DOC-02,"DOC-02, DOC-02, DOC-06",1,1.0,1
2,Какая библиотека лучше всего подходит для клас...,DOC-03,"DOC-03, DOC-09, DOC-03",1,1.0,1
3,Как расшифровывается аббревиатура RAG?,DOC-04,"DOC-05, DOC-04, DOC-04",1,1.0,2
4,Что означает термин галлюцинация в контексте LLM?,DOC-05,"DOC-05, DOC-05, DOC-02",1,1.0,1
5,Приведи пример популярного алгоритма токенизац...,DOC-06,"DOC-06, DOC-02, DOC-04",1,1.0,1
6,В чем заключается метод LoRA?,DOC-07,"DOC-07, DOC-08, DOC-04",1,1.0,1
7,Что такое Chain-of-Thought и для чего он нужен?,DOC-08,"DOC-08, DOC-08, DOC-04",1,1.0,1
8,Чем векторные базы данных отличаются от обычны...,DOC-09,"DOC-09, DOC-09, DOC-02",1,1.0,1
9,Зачем делать нахлест (overlap) при нарезке док...,DOC-10,"DOC-10, DOC-01, DOC-06",1,1.0,1



Артефакт сохранен: d:\AIE_1\homeworks\HW14\artifacts\retrieval_eval.csv


In [20]:
# --- Эксперимент с параметрами retrieval (Пункт 2.3.6) ---


def evaluate_top_k(k_value: int) -> float:
    """Вспомогательная функция для расчета Hit@K на текущем индексе."""
    hits = 0
    for query_text, expected_id in benchmark_queries:
        q_vec = embedder.encode([query_text], convert_to_numpy=True)
        faiss.normalize_L2(q_vec)
        
        dists, idxs = vector_index.search(q_vec, k_value)
        retrieved_ids = [processed_chunks[i]['parent_id'] for i in idxs[0]]
        
        if expected_id in retrieved_ids:
            hits += 1
    return hits / len(benchmark_queries)

# Проводим эксперимент
hit_rate_top1 = evaluate_top_k(1)
hit_rate_top3 = evaluate_top_k(3)

print("Сравнительный эксперимент: влияние параметра top_k на качество извлечения")
print("-" * 75)
print(f"Hit@1: {hit_rate_top1:.2f} ({hit_rate_top1*100:.0f}%)")
print(f"Hit@3: {hit_rate_top3:.2f} ({hit_rate_top3*100:.0f}%)")
print("-" * 75)

display(Markdown("""
### Вывод эксперимента
Сравнение показывает, что при извлечении строго одного документа (`top_k=1`) метрика Hit падает до 0.90. Это происходит из-за пересечения терминологии: на запрос об аббревиатуре RAG векторный поиск ставит на первую позицию чанк о "Галлюцинациях", так как он тоже содержит этот термин и обладает сильным семантическим сходством.

Расширение окна поиска до `top_k=3` позволяет захватить целевой документ со второй позиции, повышая Hit Rate до 1.00. Эксперимент доказывает необходимость использования `top_k > 1` в архитектуре RAG: передача нескольких контекстов языковой модели компенсирует неидеальность векторного ранжирования.
"""))

Сравнительный эксперимент: влияние параметра top_k на качество извлечения
---------------------------------------------------------------------------
Hit@1: 0.90 (90%)
Hit@3: 1.00 (100%)
---------------------------------------------------------------------------



### Вывод эксперимента
Сравнение показывает, что при извлечении строго одного документа (`top_k=1`) метрика Hit падает до 0.90. Это происходит из-за пересечения терминологии: на запрос об аббревиатуре RAG векторный поиск ставит на первую позицию чанк о "Галлюцинациях", так как он тоже содержит этот термин и обладает сильным семантическим сходством.

Расширение окна поиска до `top_k=3` позволяет захватить целевой документ со второй позиции, повышая Hit Rate до 1.00. Эксперимент доказывает необходимость использования `top_k > 1` в архитектуре RAG: передача нескольких контекстов языковой модели компенсирует неидеальность векторного ранжирования.


In [21]:
# --- Обновление базы знаний и переиндексация (Пункт 2.3.7) ---

with open(path_extra_kb, "r", encoding="utf-8") as f:
    new_articles = json.load(f)

print(f"Загружено {len(new_articles)} новых документов для обновления.")

new_chunks = []
for article in new_articles:
    text_fragments = create_overlapping_chunks(article["body"], window_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, fragment in enumerate(text_fragments):
        new_chunks.append({
            "chunk_id": f"{article['id']}_ch{i+1}",
            "parent_id": article['id'],
            "topic": article['topic'],
            "text": fragment
        })

updated_chunks_list = processed_chunks + new_chunks
print(f"Добавлено {len(new_chunks)} новых фрагментов. Всего в базе: {len(updated_chunks_list)}")

print("Вычисление векторов для обновленной базы...")
texts_updated = [c["text"] for c in updated_chunks_list]
updated_embeddings = embedder.encode(texts_updated, convert_to_numpy=True)
faiss.normalize_L2(updated_embeddings)

new_vector_index = faiss.IndexFlatIP(embedding_dim)
new_vector_index.add(updated_embeddings)
print(f"Новый индекс FAISS построен. Всего векторов: {new_vector_index.ntotal}\n")

update_test_queries = [
    "Что такое гибридный поиск и зачем он нужен?",
    "Для чего применяются кросс-энкодеры?",
    "В чем суть метода Self-RAG?"
]

before_after_results = []
SEARCH_K = 2

display(Markdown("### Влияние обновления базы на результаты поиска"))

for query in update_test_queries:
    q_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    
    dists_before, idxs_before = vector_index.search(q_vec, SEARCH_K)
    sources_before = [processed_chunks[i]['parent_id'] for i in idxs_before[0]]
    
    dists_after, idxs_after = new_vector_index.search(q_vec, SEARCH_K)
    sources_after = [updated_chunks_list[i]['parent_id'] for i in idxs_after[0]]
    
    has_changed = sources_before != sources_after
    
    before_after_results.append({
        "query": query,
        "before_retrieved_sources": ", ".join(sources_before),
        "after_retrieved_sources": ", ".join(sources_after),
        "changed": has_changed
    })
    
    print(f"Запрос: «{query}»")
    print(f"  [ДО обновления] Найдены: {', '.join(sources_before)}")
    print(f"  [ПОСЛЕ обновления] Найдены: {', '.join(sources_after)}")
    print("-" * 65)

df_update = pd.DataFrame(before_after_results)
update_artifact_path = ARTIFACTS_DIR / "retrieval_before_after_update.csv"
df_update.to_csv(update_artifact_path, index=False, encoding="utf-8")

print(f"Артефакт сохранен: {update_artifact_path}")

Загружено 3 новых документов для обновления.
Добавлено 10 новых фрагментов. Всего в базе: 46
Вычисление векторов для обновленной базы...
Новый индекс FAISS построен. Всего векторов: 46



### Влияние обновления базы на результаты поиска

Запрос: «Что такое гибридный поиск и зачем он нужен?»
  [ДО обновления] Найдены: DOC-09, DOC-04
  [ПОСЛЕ обновления] Найдены: DOC-12, DOC-09
-----------------------------------------------------------------
Запрос: «Для чего применяются кросс-энкодеры?»
  [ДО обновления] Найдены: DOC-10, DOC-08
  [ПОСЛЕ обновления] Найдены: DOC-12, DOC-10
-----------------------------------------------------------------
Запрос: «В чем суть метода Self-RAG?»
  [ДО обновления] Найдены: DOC-01, DOC-08
  [ПОСЛЕ обновления] Найдены: DOC-01, DOC-08
-----------------------------------------------------------------
Артефакт сохранен: d:\AIE_1\homeworks\HW14\artifacts\retrieval_before_after_update.csv


In [22]:
# --- Mini-RAG и анализ ошибок (Пункты 2.3.8 и 2.3.9) ---

def execute_mini_rag(query: str, top_k: int = 3) -> dict:
    """Простой mini-RAG: поиск релевантных фрагментов и сборка ответа."""
    q_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    dists, idxs = new_vector_index.search(q_vec, top_k)
    
    retrieved_texts = []
    source_docs = []
    
    for i in idxs[0]:
        chunk = updated_chunks_list[i]
        retrieved_texts.append(chunk['text'])
        if chunk['parent_id'] not in source_docs:
            source_docs.append(chunk['parent_id'])
            
    context_str = " ... ".join(retrieved_texts)
    
    answer = f"Исходя из найденных документов: {context_str}"
    
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": ", ".join(source_docs)
    }

rag_test_queries = [
    "Как работают трансформеры?",
    "Чем векторные базы отличаются от обычных?",
    "Как приготовить вкусный борщ?" # Специальный запрос вне домена для анализа
]

rag_results = []
print("Демонстрация работы Mini-RAG:\n")

for q in rag_test_queries:
    res = execute_mini_rag(q, top_k=2)
    rag_results.append(res)
    
    print(f"Вопрос: {res['question']}")
    print(f"Источники: {res['retrieved_sources']}")
    print(f"Ответ: {res['answer']}")
    print("-" * 65)

# Сохранение последнего артефакта (rag_examples.csv)
df_rag = pd.DataFrame(rag_results)
rag_artifact_path = ARTIFACTS_DIR / "rag_examples.csv"
df_rag.to_csv(rag_artifact_path, index=False, encoding="utf-8")

print(f"\nАртефакт сохранен: {rag_artifact_path}\n")

# Анализ ошибок для отчета
display(Markdown("""
### Анализ пограничных случаев и ошибок Mini-RAG
1. **Запросы вне домена (Out-of-Domain):** На вопрос про борщ система всё равно возвращает фрагменты про ИИ (Трансформеры/Токенизацию). Это классическая проблема векторного поиска FAISS — он всегда возвращает ближайших соседей, даже если смысловое сходство минимально. **Как исправить:** введение порога отсечения (threshold) по косинусному расстоянию или использование LLM-фильтра для предварительной оценки вопроса.
2. **Ограничения шаблонного генератора:** Текущий экстрактивный RAG просто склеивает найденные тексты. Он не умеет делать выводы или писать связные ответы своими словами. **Как исправить:** замена функции-генератора на вызов легковесной LLM (например, Llama-3 или Saiga) через API.
"""))

Демонстрация работы Mini-RAG:

Вопрос: Как работают трансформеры?
Источники: DOC-12, DOC-10
Ответ: Исходя из найденных документов: о поиска. Обычно выполняется с помощью кросс-энкодеров (cross-encoders), которые медленнее ... оделей эмбеддингов. Часто чанки делают с перекрытием (overlap).
-----------------------------------------------------------------
Вопрос: Чем векторные базы отличаются от обычных?
Источники: DOC-03, DOC-02
Ответ: Исходя из найденных документов: ва и кластеризации плотных векторов. Она содержит алгоритмы, которые ищут в наборах вектор ... зкие по смыслу тексты имеют похожие векторы (косинусное расстояние между ними стремится к 
-----------------------------------------------------------------
Вопрос: Как приготовить вкусный борщ?
Источники: DOC-10, DOC-06
Ответ: Исходя из найденных документов: оделей эмбеддингов. Часто чанки делают с перекрытием (overlap). ... Процесс разбиения сырого текста на токены: слова, части слов или символы. Современные LLM 
---------------


### Анализ пограничных случаев и ошибок Mini-RAG
1. **Запросы вне домена (Out-of-Domain):** На вопрос про борщ система всё равно возвращает фрагменты про ИИ (Трансформеры/Токенизацию). Это классическая проблема векторного поиска FAISS — он всегда возвращает ближайших соседей, даже если смысловое сходство минимально. **Как исправить:** введение порога отсечения (threshold) по косинусному расстоянию или использование LLM-фильтра для предварительной оценки вопроса.
2. **Ограничения шаблонного генератора:** Текущий экстрактивный RAG просто склеивает найденные тексты. Он не умеет делать выводы или писать связные ответы своими словами. **Как исправить:** замена функции-генератора на вызов легковесной LLM (например, Llama-3 или Saiga) через API.
